Dependencies

In [9]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import random

Setup the environment

In [18]:
env = gym.make("FrozenLake-v1",render_mode="human")

In [4]:
action_size = env.action_space.n
state_size = env.observation_space.n

Q-Table

In [5]:
qtable = np.zeros((state_size, action_size))
print(qtable)

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


### Hyperparameters
- total episodes -> so that it does not run infinitely
- maximum steps in an episode
- learning rate ($\alpha$)
- discounr rate/factor ($\gamma$)

In [6]:
total_episodes = 15000        # Total episodes
max_steps = 99                # Max steps per episode
learning_rate = 0.8           # Learning rate
gamma = 0.95                  # Discounting rate

### Exploring
- Epsilon (initially 1) ($ϵ$)
- Max value of $ϵ$ = 1 (for exploration , esp initially)
- Min value of $ϵ$ = 0.01
- Decay rate = 0.005


In [7]:
# Exploration parameters
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability
decay_rate = 0.005

### Implementing Q-Learning
- list of rewards -> initially empty
- for loop over the total episodes -> get a singular episode
  - initially reset the environment , step becomes 0 , since reset so done/game over -> 0 and the total rewards are 0 as well
- then loop over the maximum number of steps available
  - choose an action
  - use the random number
    - see if it is greater than $ϵ$ -> if yes -> exploitation
    - else random action -> exploration
  - then check the reward and the outcomes
  - update Q-value : $Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]$
  - then manage the total rewards
  - then the next state
  - check if the game is over
    - got the frizbee or fell inside the hole
      - if yes -> game over -> terminate (break the loop)
  - reduce the value of $ϵ$ using the decay we defined

- Final score : (sum of rewards)/(total number of episodes)

In [14]:
# List of rewards
rewards = []

# For each episode
for episode in range(total_episodes):
    # Reset the environment
    state, info = env.reset()  # Extract state from reset tuple
    step = 0
    done = False
    total_rewards = 0

    for step in range(max_steps):
        # Choose an action
        exp_exp_tradeoff = random.uniform(0, 1)

        # Exploitation vs exploration
        if exp_exp_tradeoff > epsilon:
            action = np.argmax(qtable[state, :])
        else:
            action = env.action_space.sample()

        # Take the action and observe the outcome
        new_state, reward, done, truncated, info = env.step(action)

        # Update Q-table
        qtable[state, action] = qtable[state, action] + learning_rate * (
            reward + gamma * np.max(qtable[new_state, :]) - qtable[state, action]
        )

        total_rewards += reward
        state = new_state  # Update state

        # If done or truncated, finish episode
        if done or truncated:
            break

    # Reduce epsilon
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    rewards.append(total_rewards)

print("Score over time: " + str(sum(rewards) / total_episodes))
print(qtable)

Score over time: 0.48306666666666664
[[2.10892463e-01 5.63882684e-02 5.56388109e-02 1.19614250e-01]
 [1.13723583e-02 1.44511765e-03 1.87125372e-02 2.15464457e-01]
 [1.23169803e-02 1.46538090e-02 1.50891764e-02 1.61148909e-01]
 [4.71499226e-03 6.17144239e-03 9.67158156e-04 2.18163208e-02]
 [3.14336803e-01 3.64368161e-02 1.26329212e-02 1.01539358e-02]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.25905362e-03 8.04522279e-09 5.79698056e-05 4.04788240e-08]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [4.61730950e-02 5.06299496e-02 1.38380723e-02 3.58431627e-01]
 [2.31239775e-02 8.00914881e-01 2.72147157e-02 2.56219425e-02]
 [2.08194175e-01 2.95306290e-02 9.99814432e-03 1.49075816e-05]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [2.24432174e-02 5.98575521e-04 6.98082276e-01 1.19418055e-01]
 [3.32432381e-01 9.97439673e-01 2.76217132e-01 2.25005811e-01]
 [0.00000000e+00 0

### Playing the frozen lake

In [19]:
# Reset environment (optional, as reset is called in the loop)
env.reset()

for episode in range(5):
    state, info = env.reset()  # Unpack the state from the reset tuple
    step = 0
    done = False
    print("****************************************************")
    print("EPISODE ", episode)

    for step in range(max_steps):
        # Take the action with the maximum expected future reward
        action = np.argmax(qtable[state, :])

        # Unpack the step tuple, including truncated
        new_state, reward, done, truncated, info = env.step(action)

        if done or truncated:  # Check both done and truncated
            # Render the environment to see the final state
            env.render()

            # Print the number of steps taken
            print("Number of steps", step)
            break
        state = new_state  # Update state

env.close()

****************************************************
EPISODE  0
Number of steps 25
****************************************************
EPISODE  1
Number of steps 27
****************************************************
EPISODE  2
Number of steps 13
****************************************************
EPISODE  3
Number of steps 98
****************************************************
EPISODE  4
Number of steps 16
